---
# 3. Exploratory Data Analysis & Visualisation

In this section, we will be conducting Data Analysis and Visualisations on the EMNIST Data provided to better understand the Dataset provided to us. Some of the Analysis will provide insights on areas to address to ensure our GAN / VAE performs better and much more consistent.

---
## 3.1 Visualising Sample Training Images (No Augmentation)

In this sub-section, we will be visualising a Sample Image for each of the class in the provided EMNIST Dataset. This will allow us to have a confirmation on what classes are there for us to predict and also ensure that the images are correctly assigned the Class Label. The visualisation will be conducted for the Non-Augmented Training Data first in the Code Cell below.

In [ ]:
# ========== Define Selected Labels and Map to Letters ========== #
selected_labels = [1, 2, 4, 5, 6, 7, 9, 10, 12, 14, 15, 16, 17, 20, 24, 26]
label_to_letter = {label: chr(64 + label) for label in selected_labels}

# ========== Plot Setup ========== #
plt.figure(figsize=(12, 28))

for row_idx, label in enumerate(selected_labels):
    matching_indices = np.where(y_train == label)[0]

    # ----- Take First 3 Samples of Label ----- #
    for col_idx in range(3):
        if col_idx >= len(matching_indices):
            continue

        sample_idx = matching_indices[col_idx]
        image = X_train[sample_idx]

        plt.subplot(len(selected_labels), 3, row_idx * 3 + col_idx + 1)
        plt.imshow(image.squeeze(), cmap='gray')
        if col_idx == 1:
            plt.title(f"{label_to_letter[label]} (Label: {label})")
        plt.axis('off')

# ========== Display Plot ========== #
plt.tight_layout()
plt.suptitle("Three Sample Images per Class (Non-Augmented)", fontsize=18, y=1.02)
plt.show()

With reference to the output above, we are able to see that the orientation of all the EMNIST Letters are in the wrong orientation. However, we are also able to identify that there is a pattern to be observed which will allow us to create an action plan to address the orientation problem. We are able to see that all the classes needs to be rotated **90 Degrees Clockwise** and **Mirrorer**. The solution will be conducted in the next section.

---
### 3.1.1 Data Orientation Resolution

In this sub-section, we will be resolving the issue identified with the orientation of the EMNIST Data Provided. We will be performing the following changes on both the Training and Validation Set. We will also be redefining the Data Augmentation Generator so as to ensure that the Augmentation is derived from the correct and intended orientation before Augmentation is introduced. The changes that will be made are indicated below:

- Rotate all Images by 90 Degrees Clockwise
- Mirror all Images from Left to Right

With the changes indicated to resolve the Data Issues, we will proceed to conduct the operation in the code cell below.

In [ ]:
# ========== Orientation Fix Function (Rotate + Mirror) ========== #
def fix_image_orientation_uniform(X_data):
    for i in range(X_data.shape[0]):
        image = X_data[i, :, :, 0]               # Shape: (28, 28)
        image = np.rot90(image, k=-1)            # Rotate 90° clockwise
        image = np.fliplr(image)                 # Mirror left-to-right
        X_data[i, :, :, 0] = image               # Save back
    return X_data

# ========== Apply Fixes to Training and Validation Sets ========== #
X_train = fix_image_orientation_uniform(X_train)
X_val = fix_image_orientation_uniform(X_val)
print("Images Rotated 90° Clockwise and Mirrored (Left-Right) Successfully.")

# ========== Redefine Data Augmentation ========== #
SEED = 42
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)
datagen.fit(X_train, seed=SEED)
augmented_generator = datagen.flow(X_train, y_train, batch_size=32, shuffle=True, seed=SEED)
print("Augmentation Pipeline Reset.")

# ========== Define Selected Labels and Map to Letters ========== #
selected_labels = [1, 2, 4, 5, 6, 7, 9, 10, 12, 14, 15, 16, 17, 20, 24, 26]
label_to_letter = {label: chr(64 + label) for label in selected_labels}

# ========== Plot: 3 Samples per Class from X_train ========== #
plt.figure(figsize=(12, 28))

for row_idx, label in enumerate(selected_labels):
    matching_indices = np.where(y_train == label)[0]

    for col_idx in range(3):
        if col_idx >= len(matching_indices):
            continue

        sample_idx = matching_indices[col_idx]
        image = X_train[sample_idx]

        plt.subplot(len(selected_labels), 3, row_idx * 3 + col_idx + 1)
        plt.imshow(image.squeeze(), cmap='gray')
        if col_idx == 1:
            plt.title(f"{label_to_letter[label]} (Label: {label})")
        plt.axis('off')

# ========== Display Plot ========== #
plt.tight_layout()
plt.suptitle("Three Sample Images per Selected Class (Rotated + Mirrored)", fontsize=18, y=1.02)
plt.show()

With reference to the visualisation above, we are able to see that the Orientation of the Data have been fixed successfully and we are able to proceed with the next EDA. We are also able to make the observation that each class contains a mixture of Capital and Small Letters. This will be left in as it can be beneficial to the GAN / VAE when it learns the various patterns. We also observe that the label range is from 1 to 26 correspondign to the EMNIST Letter's sequence. We will need to re-label to range from 0 to 15 so as to limit the negative implications during GAN / VAE Training.

---
### 3.1.2 Re-labeling Classes

In this sub-section, we will be re-labeling the current classes so as to change it from 1 to 26 to 0 to 15. We will re-label both the Training Data and the Validation Data. This will allow us to streamline the GAN / VAE Training in the future and make it easier to declare the Number of Classes rather than assuming 26. The Re-labeling map for the Current Labels that we will adhere to is indicated below:

- Label 1 (Letter A) -> Label 0
- Label 2 (Letter B) -> Label 1
- Label 4 (Letter D) -> Label 2
- Label 5 (Letter E) -> Label 3
- Label 6 (Letter F) -> Label 4
- Label 7 (Letter G) -> Label 5
- Label 9 (Letter I) -> Label 6
- Label 10 (Letter J) -> Label 7
- Label 12 (Letter L) -> Label 8
- Label 14 (Letter N) -> Label 9
- Label 15 (Letter 0) -> Label 10
- Label 16 (Letter P) -> Label 11
- Label 17 (Letter Q) -> Label 12
- Label 20 (Letter T) -> Label 13
- Label 24 (Letter X) -> Label 14
- Label 26 (Letter Z) -> Label 15

With the Re-label mapping indicated above, we will proceed with conducting the Re-label operation in the Code Cell below.

In [ ]:
# ========== Mapping (Index to Letter) ========== #
index_to_letter = {
    0: 'A', 1: 'B', 2: 'D', 3: 'E',
    4: 'F', 5: 'G', 6: 'I', 7: 'J',
    8: 'L', 9: 'N', 10: 'O', 11: 'P',
    12: 'Q', 13: 'T', 14: 'X', 15: 'Z'
}

# ========== Map Original Label to New Label ========== #
label_to_index = {label: idx for idx, label in enumerate(selected_labels)}
index_to_label = {idx: label for label, idx in label_to_index.items()}

# ========== Direct Re-label ========== #
y_train = np.array([label_to_index[y] for y in y_train])
y_val   = np.array([label_to_index[y] for y in y_val])

# ========== One-Hot Encode ========== #
y_cat_train = to_categorical(y_train, num_classes=16)
y_cat_val   = to_categorical(y_val,   num_classes=16)

# ========== Plot: 3 Samples per Class from X_train ========== #
plt.figure(figsize=(12, 28))

for row_idx in range(16):
    matching_indices = np.where(y_train == row_idx)[0]

    for col_idx in range(3):
        if col_idx >= len(matching_indices):
            continue

        sample_idx = matching_indices[col_idx]
        image = X_train[sample_idx]

        plt.subplot(16, 3, row_idx * 3 + col_idx + 1)
        plt.imshow(image.squeeze(), cmap='gray')
        if col_idx == 1:
            char = index_to_letter[row_idx]
            plt.title(f"{char} (Label: {row_idx})")
        plt.axis('off')

# ========== Display Plot ========== #
plt.tight_layout()
plt.suptitle("Three Sample Images per Selected Class (Rotated + Mirrored)", fontsize=18, y=1.02)
plt.show()

With reference to the output of the Sample Image Visualisation, we are able to verify that the Re-labelling have been executed and conducted successfully as intended. This implies that the Training Data is now corrected and will not have any negative implications during the GAN / VAE Training.

---
## 3.2 Visualising Class Distribution

In this sub-section, we will be visualising the quantity count for each of the EMNIST class so as to identify any potential Class Imbalance. Class Imbalance could cause Skewed Performance as the GAN / VAE is able to better learn from the class with higher quantity of data whereas classes with lower number of data will cause a lower quality of generated image. This can cause a GAN / VAE to be able to generate very well for a certain class but generate poorly for another class. The various actions that can be taken to address Class Imbalance are listed below:

**Methods To Address Class Imbalance**
- Establishing Class Weights
- Intense Augmentation of Minority Classes
- Oversampling / Undersampling

With the various Methods to Address Class Imbalance, we shall proceed to state the appropriate action plan for various outcomes of the visualisation. The action plan for each respective results are listed below:

**Class Imbalance Identified**
- List down the observations from the Barplot.
- Proceed to address Class Imbalance using one of the listed method.
- Justify the chosen method's usage over other method.

**No Class Imbalance Identified**
- List down the observations from the Barplot.
- Proceed to next visualisation.

With the action plan for the potential results listed, we will proceed to conduct the visualisation in the Code Cell below.

In [ ]:
# ========== Build DataFrame for Plotting ========== #
label_counts = pd.Series(y_train).value_counts().sort_index()

# ========== Remap Original Values ========== #
letters = [chr(64 + index_to_label[idx]) for idx in label_counts.index]
counts = label_counts.values

# ========== Plot Class Distribution ========== #
plt.figure(figsize=(12, 6))
ax = sns.barplot(x=letters, y=counts, hue=letters, palette='viridis', dodge=False, legend=False)

# ========== Annotate Count Above Each Bar ========== #
for i, count in enumerate(counts):
    ax.text(i, count + 50, str(count), ha='center', va='bottom', fontsize=10)

# ========== Display Plot ========== #
plt.xlabel("Character Class (Mapped from Label)")
plt.ylabel("Number of Training Samples")
plt.title("Class Distribution in Filtered EMNIST Letters (Training Set)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

With reference to the Visualisation above, we are able to see that there is a **Class Imbalance** identified within the provided EMNIST Dataset. Subsequently, we will take action to attempt to balance the EMNIST Dataset so as to ensure the results produced for each of the classes by the GAN is consistent and reliable, without creating room for deviation of performance for each of the classes.

---
### 3.2.1 Addressing Class Imbalance

As previously identified, there are Class Imbalance within the EMNIST Dataset provided. The course of action that we will undertake will be training a VAE to produce more EMNIST Data until all the Data Counts for each of the classes are equal to the highest count of data for the current classes. We will need to prepare the Encoder, Latent Sampling and Decoder before being able to generate mroe Data to balance our current EMNIST Dataset.

---
#### 3.2.1.1 Determine Class Shortfall

Before being able to utilise a VAE to address the Class Imbalance, we will firstly need to determine how much data we need for each of the classes. We will be using the current Data Count for each class and the Highest Data Count amongst the classes to determine how much data we need to bridge. A formula for this case have been indicated below:

---
Purpose: Calculate the Difference between Highest Count Class and each Class

$$
\text{Shortfall}_i = T - C_i
$$

Where:
- $T$ = Highest Data Count
- $C_i$ = Current Data Count in Class $i$
- $\text{Shortfall}_i$ = Number of Synthetic Samples Needed
---
Through the indicated Formula, we are able to determine how much Synthetic Samples we need to ensure the class are balanced. With the formula indicated, we will proceed to calculate the Class Shortfall in the Code Cell below.

In [ ]:
# ========== Calculate Current Class Frequency ========== #
label_counts = pd.Series(y_train).value_counts().sort_index()

# ========== Utilise Highest Count as Target ========== #
TARGET_PER_CLASS = label_counts.max()

# ========== Calculate Shortfall ========== #
class_shortfalls = {
    label: TARGET_PER_CLASS - count
    for label, count in label_counts.items()
    if count < TARGET_PER_CLASS
}

# ========== Create DataFrame for Shortfalls ========== #
shortfall_df = pd.DataFrame({
    'Label': list(class_shortfalls.keys()),
    'Current Count': [label_counts[label] for label in class_shortfalls.keys()],
    'Target Count': TARGET_PER_CLASS,
    'Shortfall': list(class_shortfalls.values())
})

# ========== Sort by Shortfall Descending ========== #
shortfall_df = shortfall_df.sort_values(by='Shortfall', ascending=False).reset_index(drop=True)

# ========== Display Summary ========== #
shortfall_df.style.background_gradient(cmap="Reds")

With reference with the output of the Code Cell above, we are able to determine the amount of Data that we need to generate so as to achieve a Balanced Class. We will proceed to define the Conditinal VAE's Architecture in the next section.

---
#### 3.2.1.2 Defining Callback Functions

In this sub-section, we will be pre-defining the various Callbacks. This is to ensure consistency throughout the VAE and also to increase the Conditional VAE's ability to generate data Accurately and optimise the Computation Cost for training the Conditional VAE. The formulas and logic for these Callbacks are indicated below:

---
**ReduceLROnPlateau:**

Purpose: Automatically Reduces the Learning Rate when Validation Performance Plateaus.

$$
\text{If no improvement in } val\_loss \text{ for 5 epochs: } \eta_{\text{new}} = 0.5 \cdot \eta
$$

Where:

- $\eta_{\text{new}}$ = reduced learning rate

---
**EarlyStopping:**

Purpose: Prevents Wastage of Computational Power if Model Does Not Improve.

$$
\text{If } val\_loss \text{ does not improve for 10 epochs, stop training and restore best weights}
$$

---
With the formulas and Various Callbacks listed, we will proceed to define the Callbacks in the Code Cell below in preparation for the Conditional VAE's Training.

In [ ]:
# ========== Define Early Stopping ========== #
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# ========== Define Reduce Learning Rate on Plateau ========== #
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

With reference to the Code Cell above, we are able to identify that that the Callbacks have been successfully defined and the various parameters are set so as to ensure we push the Conditional VAE to its limits in terms of the ability to generate data and also to ensure we conserve Computational Duration.

---
#### 3.2.1.3 Defining Conditional VAE Architecture

In this sub-section, we will be defining the Conditional VAE Architecture so as to generate new Data that can we added into the current Classes which fall short of the Highest Data Count as determined in the previous section. A Conditional VAE is utilised as not all Classes need to have Data Generated for the purpose of Class Balancing. THe relevant formulas for Conditional VAE are indicated below:

---
**Latent Sampling (Reparameterization Trick)**

Purpose: Enables Stochastic Sampling from the Latent Space while keeping the Model Differentiable.

$$
\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \cdot \boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon} \sim \mathcal{N}(0, \mathbf{I})
$$

Where:
- $\mathbf{z}$ = Sampled Latent Vector  
- $\boldsymbol{\mu}$ = Encoder-predicted Mean  
- $\boldsymbol{\sigma}$ = Standard Deviation (from $\log \sigma^2$)  
- $\boldsymbol{\epsilon}$ = Noise Sampled from Standard Normal Distribution

---
**KL Divergence Loss**

Purpose: Regularises latent space to Resemble a Standard Normal Distribution.

$$
\mathcal{L}_{\text{KL}} = -\frac{1}{2} \sum_{j=1}^{d} \left( 1 + \log(\sigma_j^2) - \mu_j^2 - \sigma_j^2 \right)
$$

Where:
- $d$ = Latent Space Dimensionality  
- $\mu_j$ = Mean for Dimension $j$  
- $\sigma_j$ = Standard Deviation for Dimension $j$
---
**Reconstruction Loss (Binary Crossentropy)**

Purpose: Measures how Accurately the Decoder Reconstructs Input Image.

$$
\mathcal{L}_{\text{recon}} = - \sum_{i=1}^{n} \left[ x_i \log(\hat{x}_i) + (1 - x_i) \log(1 - \hat{x}_i) \right]
$$

Where:
- $x_i$ = Original Pixel Value  
- $\hat{x}_i$ = Reconstructed Pixel  
- $n$ = Total Number of Pixels
---
**Total CVAE Loss**

Purpose: Balances Reconstruction Quality with Latent Regularisation.

$$
\mathcal{L}_{\text{CVAE}} = \mathbb{E}_{q(\mathbf{z} | \mathbf{x}, \mathbf{y})}[\mathcal{L}_{\text{recon}}] + \mathcal{L}_{\text{KL}}
$$

Where:
- $\mathbf{x}$ = Input Image  
- $\mathbf{y}$ = Class Label
- $q(\mathbf{z} | \mathbf{x}, \mathbf{y})$ = Learned Posterior Distribution of Latent Vector given Input and Label  
---
With the relevant Formulas for the Conditional VAE indicated above, we will proceed to define and consturct the Conditional VAE's Architecture in the Code Cell below so as to address the Class Imbalance.

In [ ]:
# ========== Rescale from [-1, 1] to [0, 1] ========== #
X_train_cvae = (X_train + 1.0) / 2.0
X_val_cvae = (X_val + 1.0) / 2.0

# ========== Define Hyperparameters ========== #
latent_dim = 128
num_classes = 16
img_shape = (28, 28, 1)
kl_weight = 1.0

# ========== Sampling Function ========== #
def sampling(args):
    z_mean, z_log_var = args
    eps = tf.random.normal(shape=(tf.shape(z_mean)[0], latent_dim))
    return z_mean + tf.exp(0.5 * z_log_var) * eps

# ========== VAE Loss Layer ========== #
class VAELoss(tf.keras.layers.Layer):
    def __init__(self, kl_weight=1.0, **kwargs):
        super().__init__(**kwargs)
        self.kl_weight = kl_weight
        self.bce = BinaryCrossentropy(from_logits=False, reduction='none')

    def call(self, inputs):
        x, x_decoded, z_mean, z_log_var = inputs
        recon = self.bce(x, x_decoded)
        recon_loss = tf.reduce_sum(recon, axis=[1, 2])  # [batch]
        kl = -0.5 * tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1)
        total_loss = tf.reduce_mean(recon_loss + self.kl_weight * kl)
        self.add_loss(total_loss)
        return x_decoded

# ========== CVAE Encoder ========== #
img_input = Input(shape=img_shape, name="img_input")
label_input = Input(shape=(num_classes,), name="label_input")

lbl = Dense(28 * 28, activation='relu')(label_input)
lbl = Reshape((28, 28, 1))(lbl)
lbl = GaussianNoise(0.1)(lbl)
img = GaussianNoise(0.1)(img_input)

x = Concatenate()([img, lbl])

x = Conv2D(64, 3, padding="same", kernel_initializer='he_normal')(x)
x = LeakyReLU()(x)
x = BatchNormalization()(x)
x = Conv2D(64, 3, strides=2, padding="same", kernel_initializer='he_normal')(x)
x = LeakyReLU()(x)
x = Dropout(0.2)(x)
x = BatchNormalization()(x)

x = Conv2D(128, 3, padding="same", kernel_initializer='he_normal')(x)
x = LeakyReLU()(x)
x = Conv2D(128, 3, strides=2, padding="same", kernel_initializer='he_normal')(x)
x = LeakyReLU()(x)
x = BatchNormalization()(x)

x = Flatten()(x)
x = Dense(256, activation='relu')(x)

z_mean = Dense(latent_dim, name="z_mean")(x)
z_log_var = Dense(latent_dim, name="z_log_var")(x)
z = Lambda(sampling, name="z")([z_mean, z_log_var])

# ========== CVAE Decoder ========== #
dec_input = Concatenate(name="dec_input")([z, label_input])

x = Dense(7 * 7 * 128, activation="relu", name="dense_gen")(dec_input)
x = Reshape((7, 7, 128), name="reshape_gen")(x)

x = Conv2DTranspose(128, 3, strides=2, padding="same", name="deconv1")(x)
x = LeakyReLU(name="leaky1")(x)
x = BatchNormalization(name="bn1")(x)

x = Conv2DTranspose(64, 3, strides=1, padding="same", name="deconv2")(x)
x = LeakyReLU(name="leaky2")(x)
x = BatchNormalization(name="bn2")(x)

x = Conv2DTranspose(32, 3, strides=2, padding="same", name="deconv3")(x)
x = LeakyReLU(name="leaky3")(x)
x = BatchNormalization(name="bn3")(x)

decoded = Conv2DTranspose(1, 3, padding="same", activation="sigmoid", name="decoder_output")(x)

# ========== Wrap with Loss Layer ========== #
output = VAELoss(kl_weight=kl_weight)([img_input, decoded, z_mean, z_log_var])

# ========== Compile CVAE ========== #
cvae = Model([img_input, label_input], output, name="CVAE")
cvae.compile(optimizer=Adam(learning_rate=1e-4))
cvae.summary()

With reference to the Conditional VAE's Architecture Summary output above, we are able to conclude that the Architecture have been successfully defined and we are able to proceed with the training of the Conditional VAE in the next section. Upon Training, we will be able to plot out the Training and Validation Loss Curve to determine if adjustments need to be made to the Conditional VAE's Architecture.

---
#### 3.2.1.4 Training of Conditional VAE

In this section, we will be training the Conditional VAE on the EMNIST Data that are in the Classes that need to have an increase in Data Count. We will be utilising the Callbacks and the Conditional VAE which were both respectively defined previously in Section 3.2.1.2 and 3.2.1.3. Subsequently, we will evaluate the Conditional VAE's performance using a Training and Validation Loss Curve after the training. We will be conducting the Conditional VAE Training in the Code Cell below.

In [ ]:
# ========== Train CVAE ========== #
history = cvae.fit(
    [X_train_cvae, y_cat_train],
    X_train_cvae,
    validation_data=([X_val_cvae, y_cat_val], X_val_cvae),
    epochs=200,
    batch_size=32,
    shuffle=True,
    callbacks=[early_stop, reduce_lr]
)

With reference to the output above, we are able to observe that the Conditional VAE have been successfully trained and is ready to be evaluated in the next sub-section using the Training and Validation Loss Curve. Should the Conditional VAE not perform well based on the Training and Validation Loss Curve, we will have to edit the Model's Architecture and make other neccessary adjustments.

---
#### 3.2.1.5 Plotting Training Loss for Conditional VAE

In this sub-section, we will be plotting the Learning Curve to assess the Learning Behaviour of the Conditional VAE. This plot allows us to visually determine if the Conditional VAE is Underfitting, Overfitting, or achieving an Ideal Fit during training. While the Conditional VAE is an unsupervised generative model, its ability to minimise both the reconstruction loss and KL divergence effectively indicates how well it learns the latent structure of the input data.

If the Conditional VAE is **Underfitting**, it implies that the encoder-decoder pair is not effectively capturing the underlying data distribution or reconstructing input images accurately.

If the Conditional VAE is **Overfitting**, the model may have memorised training examples and fails to generalise well to unseen variations in character structures.

An **Ideal Fit** suggests that the Conditional VAE is generalising well, balancing both reconstruction quality and latent space regularisation.

The Potential Observations and Corresponding Action Plans are outlined below:

**Underfitting Learning Curve**
- Re-tune latent dimensionality or layer complexity
- Increase the number of epochs or reduce dropout
- Revisit the KL-divergence weighting (if applicable)

**Overfitting Learning Curve**
- Enable or increase dropout
- Apply stronger regularisation (e.g. L2)
- Reduce model complexity or batch size

**Ideal Fit Learning Curve**
- Retain the current architecture and hyperparameters
- Proceed with synthetic sample generation per class
- Evaluate generation quality across all 16 classes

With the potential observations and their respective action plans indicated, we will proceed to visualise the Learning Curve in the Code Cell below.

In [ ]:
# ========== Plot CVAE Training Loss ========== #
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CVAE Training and Validation Loss Curve")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

With reference to the output of the Training and Validation Loss Curve above, we are able to determine that the Conditional VAE has achieved an **Ideal Fit** and the Conditional VAE is capable of generating quality images and also regularising the Latent Space.

---
#### 3.2.1.6 Defining Inference Decoder

In this sub-section, we will build the Inference Decoder which will be required to sample the Synthetic Data using the trained weights. The Inference Decoder is a reconstructed version of the Conditional VAE but it will not include the Encoder and also will not include an Image Input and will only have the Sampled Latent Vector and the Label as input. The Inference Decoder will be defined in the Code Cell below.

In [ ]:
# ========== Build Inference Decoder ========== #
z_input = Input(shape=(latent_dim,), name="z_input")
label_gen_input = Input(shape=(num_classes,), name="label_gen_input")
dec_input = Concatenate(name="dec_input")([z_input, label_gen_input])

# ========== Obtain CVAE Decoder Layers ========== #
x = cvae.get_layer("dense_gen")(dec_input)
x = cvae.get_layer("reshape_gen")(x)
x = cvae.get_layer("deconv1")(x)
x = cvae.get_layer("leaky1")(x)
x = cvae.get_layer("bn1")(x)
x = cvae.get_layer("deconv2")(x)
x = cvae.get_layer("leaky2")(x)
x = cvae.get_layer("bn2")(x)
x = cvae.get_layer("deconv3")(x)
x = cvae.get_layer("leaky3")(x)
x = cvae.get_layer("bn3")(x)
gen_output = cvae.get_layer("decoder_output")(x)

# ========== Define Model ========== #
decoder_model = Model([z_input, label_gen_input], gen_output, name="inference_decoder")

# ========== Execution Confirmation ========== #
print("Inference Decoder Defined")

With reference to the output above, we are able to verify that the Inference Decoder have been successfully defined and we are able to proceed to Generate Synthetic Data and Visualise the Generated Data in the next sub-sections.

---
#### 3.2.1.6 Generating Synthetic Data

In this sub-section, we will begin to generate Synthetic Data according to the Shortfall calculated beforehand. We will only generate the amount of Data required so as to prevent large sum of 'non-original' data from being added to the Training Data. We will also be evaluating the generated data afterwards to view the quality of data being input into the Training Data. We will be generating the Synthetic Data in the Code Cell below.

In [ ]:
# ========== Generate Synthetic Data ========== #
synthetic_images = []
synthetic_labels = []

for class_label, shortfall in class_shortfalls.items():
    if shortfall == 0:
        continue

    onehot = np.zeros((shortfall, num_classes))
    onehot[:, class_label] = 1

    z = np.random.normal(0, 1, size=(shortfall, latent_dim))

    generated = decoder_model.predict([z, onehot], verbose=0)
    synthetic_images.append(generated)
    synthetic_labels.append(np.full((shortfall,), class_label))

# ========== Confirmation Text ========== #
print("Synthetic Data Generated Successfully")

With reference to the output of the Code Cell above, we are able to conclude that the Synthetic Data have been successfully generated and we are able to proceed to visualising the Synthetic Data and comment on the quality of the data.

---
#### 3.2.1.7 Visualising Synthetic Data

In this sub-section, we will be visualising the generated Synthetic Data so as to check the quality of the Generated Data. It is important to note that we will not be using any metrics to evaluate the accuracy of the data. This is on the basis that our objective in this case is not for Data Accuracy but to generate new data to balance our Training Data. Therefore, the objective of this visualisation is to ensure that the Synthetic Data resembles the Class that it belongs to. We will be conducting the Visualisation in the Code Cell below.

In [ ]:
# ========== Stack Synthetic Data ========== #
X_synthetic = np.vstack(synthetic_images)
y_synthetic = np.concatenate(synthetic_labels)

# ========== Map Re-labelled Index to Original Character ========== #
selected_labels = [1, 2, 4, 5, 6, 7, 9, 10, 12, 14, 15, 16, 17, 20, 24, 26]
index_to_label = {i: label for i, label in enumerate(selected_labels)}

# ========== Plot of Synthetic Data ========== #
num_classes = len(np.unique(y_synthetic))
samples_per_class = 3

fig, axes = plt.subplots(num_classes, samples_per_class, figsize=(8, num_classes * 2))

for row_idx, label in enumerate(sorted(np.unique(y_synthetic))):
    matching_indices = np.where(y_synthetic == label)[0]

    for col_idx in range(samples_per_class):
        ax = axes[row_idx, col_idx]
        ax.axis('off')

        if col_idx < len(matching_indices):
            sample_idx = matching_indices[col_idx]
            image = X_synthetic[sample_idx]
            ax.imshow(image.squeeze(), cmap='gray')

        if col_idx == 1:
            original_label = index_to_label[label]
            char = chr(64 + original_label)
            ax.set_title(f"{char} (Class {label})", fontsize=10)

# ========== Display Plot ========== #
fig.suptitle("Three Synthetic Samples per Class (CVAE)", fontsize=18, y=1.02)
plt.subplots_adjust(top=0.96, hspace=0.5)
plt.show()

With reference to the output of the Visualisation of the Synthetic Data above, we are able to see that the Synthetic Data are generally well generated. However, there are a few 'odd-looking' images but these images are minimal and will most likely not affect the training of the GAN. We are also able to observe that each class has at least 1 Synthetic Data with exception of the highest count class. With these observation, we will proceed to Stack the Synthetic Data with the Training Data to address the Class Imbalance.

---
#### 3.2.1.8 Data Stacking

In this sub-section, we will be conducting the Data Stacking operation for our Training Data. We will be combining the Synthetic Data and the Actual EMNIST Data which will subsequently address the Class Imbalance. Afterwards, we will once again Visualise the Class Distribution to verify that there is no longer any Class Imbalance within our Training Data. The Data Stacking will be done in the Code Cell below.

In [ ]:
# ========== Overwrite Training Data with Balanced Data ========== #
X_train = np.vstack([X_train, X_synthetic])
y_train = np.concatenate([y_train, y_synthetic])

# ========== One-Hot Encode ========== #
y_cat_train = to_categorical(y_train, num_classes=16)

# ========== Calculate Label Distribution ========== #
label_counts = pd.Series(y_train).value_counts().sort_index()

# ========== Map to EMNIST Dataset ========== #
selected_labels = [1, 2, 4, 5, 6, 7, 9, 10, 12, 14, 15, 16, 17, 20, 24, 26]
label_to_letter = {i: chr(64 + selected_labels[i]) for i in range(len(selected_labels))}

# ========== Map Index to Letters ========== #
letters = [label_to_letter[label] for label in label_counts.index]
counts = label_counts.values

# ========== Plot Class Distribution ========== #
plt.figure(figsize=(12, 6))
ax = sns.barplot(x=letters, y=counts, hue=letters, palette="viridis", dodge=False, legend=False)

# ========== Annotate Count Above Each Bar ========== #
for i, count in enumerate(counts):
    ax.text(i, count + 50, str(count), ha='center', va='bottom', fontsize=10)

# ========== Display Plot ========== #
plt.xlabel("Character Class (Mapped from Label)")
plt.ylabel("Number of Training Samples")
plt.title("Balanced Class Distribution")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

With reference to the output of the Updated Class Distribution, we are able to observe that all the classes have now been balanced at around 3000 Data per class. This implies that the Conditional VAE have successfully created Synthetic Data and we are ready to continue with the next Exploratory Data Analysis.

---
## 3.3 Visualising Sample Validation Images

In this sub-section, we will be visualising a Sample Image for each of the class in the provided MNIST Dataset. This will allow us to have a confirmation on whether the Data Cleaning processes done of the Training EMNIST Data have been also executed on the Validation Data. The visualisation will be conducted for the Validation Data in the Code Cell below.

In [ ]:
# ========== Map Label to Letter ========== #
index_to_letter = {
    0: 'A', 1: 'B', 2: 'D', 3: 'E',
    4: 'F', 5: 'G', 6: 'I', 7: 'J',
    8: 'L', 9: 'N', 10: 'O', 11: 'P',
    12: 'Q', 13: 'T', 14: 'X', 15: 'Z'
}

# ========== Plot of Validation Data ========== #
plt.figure(figsize=(12, 28))
for row_idx, label in enumerate(sorted(np.unique(y_val))):
    matching_indices = np.where(y_val == label)[0]
    for col_idx in range(3):
        if col_idx >= len(matching_indices):
            continue
        sample_idx = matching_indices[col_idx]
        image = X_val[sample_idx]
        plt.subplot(len(np.unique(y_val)), 3, row_idx * 3 + col_idx + 1)
        plt.imshow(image.squeeze(), cmap='gray')
        if col_idx == 1:
            char = index_to_letter.get(label, "Unknown")
            plt.title(f"{char} (Label {label})")
        plt.axis('off')

# ========== Display Plot ========== #
plt.tight_layout()
plt.suptitle("Three Sample Images per Class (Validation Set)", fontsize=18, y=1.02)
plt.show()

With reference to the output above, we are able to confirm that the Validation Dataset is in the correct orientation and all the classes are present.

---
## 3.4 Visualising Class Distribution for Validation Set

In this sub-section, we will be visualising the Class Distribution for each of the classes in the Validation Set to determine if there are any significant Class Imbalance. However, unless the Class Imbalance is Extremely Significant, we will not address the Class Imbalance as the Validation Data does not directly impact the GAN / VAE Training except for the metrics. We will be conducting the observation in the Code Cell below.

In [ ]:
# ========== Map Label to Letter ========== #
index_to_letter = {
    0: 'A', 1: 'B', 2: 'D', 3: 'E',
    4: 'F', 5: 'G', 6: 'I', 7: 'J',
    8: 'L', 9: 'N', 10: 'O', 11: 'P',
    12: 'Q', 13: 'T', 14: 'X', 15: 'Z'
}

# ========== Validation Class Histogram ========== #
val_label_counts = pd.Series(y_val).value_counts().sort_index()
val_letters = [index_to_letter[i] for i in val_label_counts.index]

plt.figure(figsize=(10, 5))
barplot = sns.barplot(x=val_letters, y=val_label_counts.values, palette='mako')

# ===== Add Value Labels Above Each Bar ===== #
for idx, val in enumerate(val_label_counts.values):
    barplot.text(idx, val + 2, str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')

# ========== Display Plot ========== #
plt.xlabel("Character Class")
plt.ylabel("Number of Validation Samples")
plt.title("Class Distribution in Validation Set")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

With reference to the output above, we are able to observe that there is indeed Class Imbalance within the Validation Dataset. However, these imbalance only deviate from the highest value by a small margin and does not neccesarily imply it will have a negative impact on the Training of the GAN / VAE. Addressing the Class Imbalance will also most likely cause more negative than positive side effects as Synthetic Data is not Genuine Data and will most likely cause a inaccurate Loss count. Therefore, the Class Imbalance will not be addressed in the Validation Data.

---
## 3.5 Visualising Sample Training Images (Augmented Data)

In this sub-section, we will be visualising a Sample Image for each of the class in the provided MNIST Dataset that are in the Training Set and with Augmentation applied. This will allow us to have a confirmation on whether the Augmentation using the Image Data Generator has any effects on the Training Images and also to confirm that the Data Cleaning process applied to the Non-Augmented Training Data have been successfully applied here too. The visualisation will be conducted for the Non-Augmented Training Data first in the Code Cell below.

In [ ]:
# ========== Define Augmentation Generator ========== #
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=10,
    zoom_range=0.1
)

# ========== Prepare 3 Images per Class ========== #
per_class_images = []
per_class_labels = []

for class_label in range(16):
    indices = np.where(y_train == class_label)[0][:3]
    per_class_images.append(X_train[indices])
    per_class_labels.extend([class_label] * len(indices))

X_vis = np.vstack(per_class_images)

# ========== Generate Augmented Images ========== #
aug_iter = datagen.flow(X_vis, batch_size=len(X_vis), shuffle=False)
augmented_images = next(aug_iter)

# ========== Define Label Mapping ========== #
index_to_letter = {
    0: 'A', 1: 'B', 2: 'D', 3: 'E',
    4: 'F', 5: 'G', 6: 'I', 7: 'J',
    8: 'L', 9: 'N', 10: 'O', 11: 'P',
    12: 'Q', 13: 'T', 14: 'X', 15: 'Z'
}

# ========== Plot Augmented Images ========== #
plt.figure(figsize=(12, 28))

for row_idx, class_label in enumerate(range(16)):
    matching_indices = [i for i, lbl in enumerate(per_class_labels) if lbl == class_label]
    for col_idx, sample_idx in enumerate(matching_indices):
        plt.subplot(16, 3, row_idx * 3 + col_idx + 1)
        plt.imshow(augmented_images[sample_idx].squeeze(), cmap='gray')
        if col_idx == 1:
            char = index_to_letter[class_label]
            plt.title(f"{char} (Label {class_label})")
        plt.axis('off')

# ========== Display Plot ========== #
plt.suptitle("Three Augmented Images per Class (ImageDataGenerator)", fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

With reference to the output of the visualisation above, we are able to confirm that the Sample Images does show signs of Augmentation such as rotation and zoom. We are also able to verify that the Labels are correct with reference to the Re-Labelling operation done beforehand.

---
## 3.6 Visualising Average Image Heatmap Per Class

In this sub-section, we will be visualising the Average Image Heatmap for each of the EMNIST Classes provided. This will show us the general image of each classes and also the potential deviation that each class may have between Capital Letters and Small Letters. We will also be able to evaluate the clarity of each EMNIST Class as well as try to predict which image will be clearer for the output of the GAN during the Evaluation. We will also be able to determine which classes are most likely to have the lower quality output based off this Visualisation. We will be recording the observations and conducting the Visualisation in the Code Cell below.

In [ ]:
# ========== Map Label to Letter ========== #
char_map = {i: chr(64 + label) for i, label in enumerate(selected_labels)}

# ========== Calculate Average Image ========== #
avg_images = []
char_labels = []

for label in sorted(np.unique(y_train)):
    imgs = X_train[y_train == label]
    avg_img = imgs.mean(axis=0).squeeze()
    avg_images.append(avg_img)
    char_labels.append(char_map[label])

# ========== Form 2D Matrix ========== #
avg_stack = np.stack(avg_images)

# ========== Plot Heat Map ========== #
fig, axes = plt.subplots(nrows=1, ncols=len(char_labels), figsize=(2.2 * len(char_labels), 3))

for i, ax in enumerate(axes):
    sns.heatmap(avg_stack[i], ax=ax, cmap="YlGnBu", cbar=False)
    ax.set_title(char_labels[i], fontsize=14)
    ax.axis('off')

# ========== Display Plot ========== #
fig.suptitle("Average Image Intensity by Character Class", fontsize=18)
plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.show()

With reference to the output above, we are able to make the following observations:

**A**

- Very Rounded Loop at the Bottom Left which Resembles Lowercase 'a'.

- Mildly Ambiguous and Capital 'A' Not Present.

- Moderate GAN Clarity Expected.

**B**

- Strong Looped Structure.

- Clear Distinction Between Upper and Lower Loops.

- High-Level GAN Clarity Expected.

**D**

- Straight Vertical Stem with a Rightward Loop, Resembling Lowercase 'd'.

- Well-defined Shape and Moderate Deviation Between Forms.

- Moderate GAN Clarity Expected.

**E**

- Very round, Resembling Lowercase 'e'.

- Compact Structure and Very Blurred in the Center.

- Low-Level GAN Clarity Expected.

**F**

- Straight Upper Bar with a Descending Curve, Likely Lowercase 'f'.

- Good Vertical Alignment and Form is Distinct.

- High-Level GAN Clarity Expected.

**G**

- Hybrid of Uppercase 'G' and Lowercase 'g'.

- Some Ambiguity in the Loop; Mildly Unclear.

- Moderate GAN Clarity Expected.

**I**

- Very Thin Vertical Stroke, Likely Both Capital 'I' and Lowercase 'i'.

- Lacks Defining Horizontal Bars of Uppercase 'I'.

- High Confusion Risk, May Yield Low GAN Clarity.

**J**

- Clear Hook Shape at the Bottom which Resembles Lowercase 'j'.

- Loop On Top Suggests Good Variation Learning.

- High-Level GAN Clarity Expected.

**L**

- Slight Hook at Bottom and Vertical Stem, Most Likely Lowercase 'l'.

- Capital L Features not Strongly Present.

- Good Clarity but Possible Confusion with I.

**N**

- Highly Angular which Looks Like Uppercase 'N' Merged with Lowercase 'n'.

- Moderate Consistency and Decent Top-to-bottom Intensity.

- Moderate GAN Clarity Expected.

**O**

- Perfectly Rounded Loop and Very Symmetric.

- Most Visually Uniform and Clean Class.

- High-Level GAN Clarity Expected.

**P**

- Strong Vertical Stem with Distinct Loop.

- No Distinction Uppercase and Lowercase.

- High-Level GAN Clarity Expected.

**Q**

- Slight Tail on Bottom-right and Loop Resembles 'a' / 'g'.

- Lowercase 'q' Effect Visible.

- Moderate GAN Clarity Expected.

**T**

- Crossbar Visible and Vertical Stroke Consistent.

- Balanced Between Uppercase and Lowercase Styles.

- Likely Strong GAN Clarity.

**X**

- Very Symmetric and Clean.

- Distinct Cross Structure.

- High-Level GAN Clarity Expected.

**Z**

- Angular zigzag pattern; well-defined.

- Extremely Consistent Shape.

- High-Level GAN Clarity Expected.

With the observations and implications indicated above, we will proceed to the next Exploratory Data Analysis and Visualisation in the next sub-section.

---
## 3.7 Visualising Pixel-wise Variance Heatmap

In this sub-section, we will be Visualising the Pixel-wise Variance using a Heatmap. Through this visualisation, we will be able to observe the various regions, mainly the bright and dark regions across the Training Samples. This will assist us in indentifying which parts of the Image are Key Features for the GAN to learn from. The Visualisation will be done in the Code Cell below.

In [ ]:
# ========== Compute Pixel-wise Variance  ========== #
pixel_var = np.var(X_train, axis=0).squeeze()

# ========== Plot Variance Heatmap ========== #
plt.figure(figsize=(8, 8))
ax = sns.heatmap(
    pixel_var,
    cmap="RdBu_r",
    cbar=True,
    square=True,
    xticklabels=4,
    yticklabels=4,
    linewidths=0.3,
    linecolor='gray'
)

# ========== Display Plot ========== #
plt.title("Pixel-wise Variance Heatmap", fontsize=18)
plt.xlabel("Pixel X-axis")
plt.ylabel("Pixel Y-axis")
plt.xticks(fontsize=8)
plt.yticks(fontsize=8)
plt.axis('on')
plt.tight_layout()
plt.show()

With reference to the output of the Visualisation in the Code Cell above, we are able to make the following observations:

**High Variance in Central Region**
- The Central Pixel Area (~8-20 on Both Axes) Exhibits the Highest Variance.

- Indicates that this is where the Majority of Handwriting Activity Occurs.

**Low Variance at Borders**
- Border Pixels (Near Edges) Show Nearly Zero Variance.

- This implies they Consistently Remain Dark and Unused Across Samples.

**Rounded High-Variance Shape**
- The Pattern of Variance Forms a Rounded Square.

- Aligns with Typical Handwriting Distribution on the EMNIST Canvas.

**Potential Character Stroke Zones**
- Areas of High Variance Likely Correspond to where Letter Strokes Begin.

**Consistency Across Classes**
- Despite Class Diversity, Aggregated Variance Heatmap Suggests a Consistent Writing Region Across All EMNIST Characters.

With the observations from the Visualisation indicated above, we will proceed with conducting the next Exploratory Data Analysis / Visualisation in the next sub-section.

---
## 3.8 Heatmap Visualisation Between Class Correlation

In this sub-section, we will be Visualising a Heatmap to see the Class Correlation between the various Classes. This will allow us to foresee potential mix up between certain Classes and also give us the ability to justify why these mix-ups may occur. Subsequently, we are also able to see which Classes will have the least mix-ups. The Visualisation will be conducted in the Code Cell below.

In [ ]:
# ========== Flatten Image ========== #
flat_images = X_train.reshape(len(X_train), -1)
df_flat = pd.DataFrame(flat_images)
df_flat['label'] = y_train

# ========== Compute Mean Pixel Intensity per Class ========== #
char_mean_vectors = df_flat.groupby('label').mean()

# ========== Correlation Matrix ========== #
corr_matrix = np.corrcoef(char_mean_vectors)

# ========== Extract Labels ========== #
char_labels = list(char_map.values())

# ========== Plot Heatmap Correlation ========== #
sns.clustermap(
    corr_matrix,
    cmap='coolwarm',
    xticklabels=char_labels,
    yticklabels=char_labels,
    figsize=(24, 13),
    linewidths=0.5,
    annot=True,
    fmt=".2f"
)

# ========== Display Plot ========== #
plt.title("Correlation Between Character Classes Based on Mean Pixel Intensities", fontsize=16)
plt.xlabel("Character Class", fontsize=12)
plt.ylabel("Character Class", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

With reference to the output of the Heatmap Visualisation above, we are able to make the following observations:

**High Correlation Groups (Potential Mix-ups)**

- 'P', 'F', and 'E' show Very High Correlation (above 0.88), indicating these classes share Similar Structural Patterns such as Vertical Strokes and Horizontal Bars.

- 'I', 'J', and 'L' are Highly Similar (J-L: 0.96 correlation), suggesting likely confusion due to Simple Straight-line Designs.

- 'T', 'Z', and 'X' Cluster Moderately Well, likely due to Symmetrical or Cross-like Shapes.

- 'B' and 'D' exhibit Moderate Correlation (0.77), both involving Round Loops attached to Vertical Stems.

**Distinct or Low-Correlation Characters (Least Likely to be Mixed)**

- 'N' has Very Low Correlation with all other characters (as low as 0.23), indicating it has a Highly Unique Diagonal Structure and is unlikely to be Misclassified.

- 'G' and 'Q' also exhibit Low Correlation with Several Characters, showing Distinctiveness in their Looped and Tail-like Shapes.

**Surprising Pairs**

- 'A' and 'O' have Moderate Correlation (0.63), although Structurally Different, likely due to their Centralised, Rounded Bases.

- 'G' shares Moderate Similarity with C-like Shapes such as 'O' and 'Q'.

**Cluster Insights**

- The Hierarchical Clustering clearly Groups Characters based on Average Stroke Patterns, enabling Early Identification of High-Risk Confusion Clusters for Classifier Training and Evaluation.

- Some Class Pairs (e.g. I vs J, P vs F) may require Additional Augmentation or Contrastive Learning Strategies to Enhance Separation.

With the observations and implications indicated above, we will proceed to conduct the next Exploratory Data Analysis in the following sub-section.

---
## 3.9 Pixel Intensity Distribution Curve

In this sub-section, we will be visualising the Distribution of the Pixel Intensity. Through this visualisation, we will be able to determine if there are any outliers within the provided EMNIST Dataset. We will also be able to determine the various Frequency for the various Pixels of the Sample Images. The Distribution Curve will be plotted in the Code Cell below.

In [ ]:
# ========== Calculate Mean and Standard Deviation ========== #
means = X_train.mean(axis=(1, 2, 3))
stds = X_train.std(axis=(1, 2, 3))

plt.figure(figsize=(14, 6))

# ========== Plot Mean Pixel Intensity Distribution ========== #
plt.subplot(1, 2, 1)
sns.histplot(means, kde=True, color='royalblue', bins=30, edgecolor='black')
plt.title("Distribution of Mean Pixel Intensities", fontsize=14, weight='bold')
plt.xlabel("Mean Pixel Intensity", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.3)

# ========== Plot Standard Deviation Pixel Intensity Distribution ========== #
plt.subplot(1, 2, 2)
sns.histplot(stds, kde=True, color='darkorange', bins=30, edgecolor='black')
plt.title("Distribution of Pixel Intensity Standard Deviation", fontsize=14, weight='bold')
plt.xlabel("Standard Deviation of Pixel Intensity", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.3)

# ========== Display Plots ========== #
plt.suptitle("Sample-Level Pixel Intensity Statistics", fontsize=16, weight='bold', y=1.03)
plt.tight_layout()
plt.show()

With reference to the output above, we are able to make the following observations:

**Gaussian-like Distribution for Both Metrics**

- The mean Pixel Intensities and Standard Deviations both form Bell Curves.

- This suggests a Healthy Distribution without Extreme Outliers or Anomalies.

**Mean Intensity Clustered Around ~0.18**

- Most EMNIST samples have Low-to-moderate Overall Brightness
- Consistent with Grayscale Characters on Dark Backgrounds.

**Standard Deviation Peaks Near ~0.33**

- Indicates that the Majority of Characters show Moderate Variation in Pixel Values, Typical of Handwritten Shapes.

**No Apparent Skew or Extreme Tails**

- The Distributions are Symmetric, Indicating Good Dataset Uniformity and no need for Intensity Normalisation or Filtering.

**Good Indicator of GAN Stability**

- These Uniform Stats Imply that Models like GANs or VAEs will have a Consistent Pixel Scale to learn from, Aiding Convergence.

With the observations indicated above, we will proceed with the next Exploratory Data Analysis / Visualisation in the next sub-section.

---
## 3.10 t-SNE Projection by Character Class

In this sub-section, we will be using t-SNE to reduce the dimensions of the current Data Points, allowing us to visualise the Natural Groupings and Clusters within the provided EMNIST Dataset. We are also able to Evaluate the Class Separability and also identify the various Outliers for each class. The t-SNE Projection will be conducted in the Code Cell below.

In [ ]:
# ========== Flatten Image ========== #
flat = X_train.reshape((X_train.shape[0], -1))

# ========== Apply t-SNE on Data ========== #
tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42)
X_embedded = tsne.fit_transform(flat)

# ========== Store t-SNE Data in DataFrame ========== #
df_tsne = pd.DataFrame({
    'x': X_embedded[:, 0],
    'y': X_embedded[:, 1],
    'label': y_train
})

# ========== Map Data Labels ========== #
df_tsne['char'] = df_tsne['label'].map(char_map)

# ========== Plot t-SNE Projection ========== #
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df_tsne, x='x', y='y', hue='char', palette='tab20', s=20)

# ========== Display Plot ========== #
plt.title("t-SNE Projection by Character Class", fontsize=16)
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.legend(title='Character', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

With reference to the output of the visualisation of the t-SNE Visualisation, we are capable to make the following observations:

**Strong Clustering**

- Letters like 'A', 'Z', 'X', 'N', 'P', 'D', and 'E' form Tight and Distinct Clusters, indicating Consistent Handwriting Patterns and Clear Separability.

- These Classes are Less Likely to be Misclassified.

**Moderate Overlap**

- 'B', 'Q', 'O', and 'G' share Visual Space with Several Clusters.

- These Characters often have Rounded Shapes, leading to Higher Confusion.

- Clusters like 'O' / 'Q' and 'B' / 'G' tend to Merge, Indicating Potential Misclassification Risk.

**High Dispersion**

- The Class 'I' is Highly Scattered, likely due to Variations in how people write 'I' (some as a Single Line, others with Serifs or Curves).

- This class may be harder to model.

**Unexpected Proximity**

- Some classes like 'F' and 'T' show Unexpected Adjacency, which may Reflect Stylistic Similarities in some Handwriting Samples.

**Character Shape Dominance**

- The Embedding seems to group letters by Shape (e.g. curved vs. straight).

- This suggest that t-SNE captured Semantic Pixel-level Similarities.